# Домашняя работа 8. Публичное API и база данных PostgreSQL

Две независимые части: сбор данных из An API of Ice and Fire (`https://anapioficeandfire.com`) и запросы к публичной базе RNAcentral.

# Часть 1. Публичное API

## 1. Информация обо всех книгах (1 балл)

In [1]:
"""Задача 1. Информация обо всех книгах из An API of Ice and Fire.

API отдаёт JSON и не требует авторизации: документация —
https://github.com/joakimskoog/AnApiOfIceAndFire/wiki. Максимальный размер
страницы у него 50 записей, поэтому данные придётся листать.
"""

import pandas as pd
import requests

API = "https://anapioficeandfire.com/api"
PAGE_SIZE = 50          # больше 50 записей за один запрос API не отдаёт

session = requests.Session()
session.headers.update({"Accept": "application/json"})


def fetch_all(resource: str, **filters) -> list[dict]:
    """Забирает все страницы ресурса и возвращает их одним списком.

    Общее число страниц API в теле ответа не сообщает — он отдаёт ссылки
    в заголовке Link, и признаком продолжения служит наличие в нём rel="next".
    Поэтому листаем не «пока номер страницы меньше N», а пока API сам
    предлагает следующую страницу.
    """
    records: list[dict] = []
    page = 1
    while True:
        response = session.get(f"{API}/{resource}",
                               params={**filters, "page": page, "pageSize": PAGE_SIZE},
                               timeout=30)
        response.raise_for_status()
        batch = response.json()
        if not batch:
            break

        records.extend(batch)
        if 'rel="next"' not in response.headers.get("Link", ""):
            break
        page += 1

    return records


books = pd.DataFrame(fetch_all("books"))

print(f"книг получено: {len(books)}")
print(f"столбцов:      {books.shape[1]}")
print(f"названия:      {list(books.columns)}")
print()
books[["name", "authors", "numberOfPages", "publisher", "released"]]

книг получено: 12
столбцов:      11
названия:      ['url', 'name', 'isbn', 'authors', 'numberOfPages', 'publisher', 'country', 'mediaType', 'released', 'characters', 'povCharacters']



,name,authors,numberOfPages,publisher,released
0,A Game of Thrones,[George R. R. Martin],694,Bantam Books,1996-08-01T00:00:00
1,A Clash of Kings,[George R. R. Martin],768,Bantam Books,1999-02-02T00:00:00
2,A Storm of Swords,[George R. R. Martin],992,Bantam Books,2000-10-31T00:00:00
3,The Hedge Knight,[George R. R. Martin],164,Dabel Brothers Publishing,2005-03-09T00:00:00
4,A Feast for Crows,[George R. R. Martin],784,Bantam Books,2005-11-08T00:00:00
5,The Sworn Sword,[George R. R. Martin],152,Marvel,2008-06-18T00:00:00
6,The Mystery Knight,[George R. R. Martin],416,Tor Fantasy,2011-03-29T00:00:00
7,A Dance with Dragons,[George R. R. Martin],1040,Bantam Books,2011-07-12T00:00:00
8,The Princess and the Queen,[George R. R. Martin],784,Tor Books,2013-12-03T00:00:00
9,The Rogue Prince,[George R. R. Martin],832,Bantam Books,2014-06-17T00:00:00


**О пагинации.** Общее число книг — 12, они помещаются в одну страницу (`pageSize=50`).
Но у ресурса «дома» записей 444, и там листание обязательно: 9 страниц по 50 записей.
Признак того, что есть следующая страница, API передаёт в заголовке `Link` — в теле ответа
номера последней страницы нет.

**Про подсказку из задания.** Для фильтрации используется параметр `hasTitles`
(по документации такие параметры называются фильтрами и принимают `true`/`false`).

## 2. Информация обо всех домах Вестероса (1 балл)

In [2]:
houses = pd.DataFrame(fetch_all("houses"))

print(f"домов Вестероса получено: {len(houses)}")
print(f"столбцов:                 {houses.shape[1]}")
print()
houses[["name", "region", "words", "coatOfArms", "founded"]].head(10)

домов Вестероса получено: 444
столбцов:                 16



,name,region,words,coatOfArms,founded
0,House Algood,The Westerlands,,"A golden wreath, on a blue field with a gold b...",
1,House Allyrion of Godsgrace,Dorne,No Foe May Pass,"Gyronny Gules and Sable, a hand couped Or",
2,House Amber,The North,,,
3,House Ambrose,The Reach,Never Resting,"Or, semy of ants gules",
4,House Appleton of Appleton,The Reach,,"Or, an apple tree eradicated proper fructed gu...",
5,House Arryn of Gulltown,The Vale,,,
6,House Arryn of the Eyrie,The Vale,As High as Honor,A sky-blue falcon soaring against a white moon...,Coming of the Andals
7,House Ashford of Ashford,The Reach,Our Sun Shines Bright,"Tenny, a sun in splendour beneath a chevron in...",
8,House Ashwood,The North,,,
9,House Baelish of Harrenhal,The Riverlands,,"A field of silver mockingbirds, on a green fie...",299 AC


**Про пустые поля.** У части домов не заполнены `words` (девиз), `region` (регион) и `founded`
(год основания) — это не ошибка разбора, а особенности данных: у младших домов девиза в книгах
просто нет. Дальше на это опирается пункт 3.

## 3. Дома Вестероса, у которых есть девиз (2 балла)

In [3]:
houses_with_motto = pd.DataFrame(fetch_all("houses", hasWords="true"))

print(f"домов с девизом: {len(houses_with_motto)}")
print(f"пустых значений в столбце words: {(houses_with_motto['words'] == '').sum()}")
print()
houses_with_motto[["name", "region", "words"]].head(10)

домов с девизом: 68
пустых значений в столбце words: 0



,name,region,words
0,House Allyrion of Godsgrace,Dorne,No Foe May Pass
1,House Ambrose,The Reach,Never Resting
2,House Arryn of the Eyrie,The Vale,As High as Honor
3,House Ashford of Ashford,The Reach,Our Sun Shines Bright
4,House Baratheon of Storm's End,The Stormlands,Ours is the Fury
5,House Beesbury of Honeyholt,The Reach,Beware our Sting
6,House Bolton of the Dreadfort,The North,Our Blades are Sharp
7,House Buckwell of the Antlers,The Crownlands,Pride and Purpose
8,House Bulwer of Blackcrown,The Reach,Death Before Disgrace
9,House Caron of Nightsong,The Stormlands,No Song so Sweet


**Почему `hasWords`, а не `hasTitles`.** Подсказка в задании предлагает фильтр `hasTitles`,
но он отбирает дома с заполненными **титулами** (`titles` — например, «Lord of the Eyrie»),
а девиз лежит в отдельном поле `words`. Девизу соответствует фильтр `hasWords`.
Разница в числах видна на следующей ячейке: она запускает оба фильтра и показывает, сколько
домов из выборки по титулам на самом деле без девиза.

In [4]:
houses_with_titles = pd.DataFrame(fetch_all("houses", hasTitles="true"))

print(f"hasWords=true  (есть девиз):   {len(houses_with_motto):>3} домов, "
      f"пустых девизов {(houses_with_motto['words'] == '').sum()}")
print(f"hasTitles=true (есть титулы):  {len(houses_with_titles):>3} домов, "
      f"пустых девизов {(houses_with_titles['words'] == '').sum()}")
print(f"всего домов в API:             {len(houses):>3}")

hasWords=true  (есть девиз):    68 домов, пустых девизов 0
hasTitles=true (есть титулы):  165 домов, пустых девизов 124
всего домов в API:             444


# Часть 2. Работа с базой данных PostgreSQL

**Соединение закрывается в конце работы** (пункт 8): курсор и подключение держат сетевой сокет
и место в пуле подключений на стороне сервера.

In [5]:
import time

import psycopg2

DB_PARAMS = {
    "host": "hh-pgsql-public.ebi.ac.uk",
    "port": 5432,
    "dbname": "pfmegrnargs",
    "user": "reader",
    "password": "NWDMCE5xdipIjRrp",
    "connect_timeout": 20,
}


def connect(attempts: int = 5) -> psycopg2.extensions.connection:
    """Подключается к публичной базе, повторяя попытку при отказе.

    База общая для всех желающих, и в загруженные часы она отвечает
    «remaining connection slots are reserved for roles with the SUPERUSER
    attribute»: свободных слотов подключения нет. Это временное состояние,
    поэтому делаем несколько попыток с нарастающей паузой.
    """
    for attempt in range(1, attempts + 1):
        try:
            return psycopg2.connect(**DB_PARAMS)
        except psycopg2.OperationalError as error:
            if attempt == attempts:
                raise
            print(f"попытка {attempt}/{attempts} не удалась: "
                  f"{str(error).strip().splitlines()[0]}")
            time.sleep(5 * attempt)


connection = connect()
parameters = connection.get_dsn_parameters()

cursor = connection.cursor()
cursor.execute("SELECT version(), current_user, current_database()")
server_version, user, database = cursor.fetchone()

print("подключение установлено")
print(f"сервер:  {parameters['host']}")
print(f"база:    {database}")
print(f"пользо:  {user}")
print(f"версия:  {server_version.split(',')[0]}")

# Проверяем, что роль видит схему с данными: у публичного пользователя доступ
# выдан только на чтение и только к схеме rnacen
cursor.execute("SELECT has_schema_privilege('rnacen', 'USAGE')")
schema_available = cursor.fetchone()[0]
print(f"\nдоступ на чтение схемы rnacen: {schema_available}")

подключение установлено
сервер:  hh-pgsql-public.ebi.ac.uk
база:    pfmegrnargs
пользо:  reader
версия:  PostgreSQL 16.11 on x86_64-pc-linux-gnu

доступ на чтение схемы rnacen: False


**Проверка доступа.** Одна строка в предыдущей ячейке проверяет, что у публичной роли `reader`
есть право `USAGE` на схему `rnacen`, в которой лежат таблицы. Это не формальность: 09.10.2026
право на стороне EBI пропало — соединение проходит, а любой запрос к таблицам возвращает
`relation "rnc_database" does not exist` и `permission denied for schema rnacen`. Ноутбук ниже
это не скрывает: он печатает сообщение сервера вместо ошибки, и те же ячейки отдают данные без
правок, как только доступ восстановят.

## 5. Десять строк из таблицы `rnc_database` (2 балла)

In [6]:
def run_query(sql: str):
    """Выполняет запрос и возвращает строки, имена столбцов и ошибку запроса.

    Публичная база живёт на сервере EBI и открыта всем желающим, поэтому запрос
    может не пройти по причинам, к нашей программе отношения не имеющим: кончились
    слоты подключений, у роли забрали права на схему, сервер обновляется. Чтобы
    ноутбук в таком случае не падал на середине, ошибка сервера возвращается
    третьим значением и печатается сообщением.
    """
    try:
        cursor.execute(sql)
        rows = cursor.fetchall()
        return rows, [column[0] for column in cursor.description], None
    except psycopg2.Error as error:
        connection.rollback()
        print(f"запрос не выполнен: {type(error).__name__}")
        print(f"сообщение сервера: {str(error).strip().splitlines()[0]}")
        return [], [], error


rows, columns, db_error = run_query("SELECT * FROM rnc_database LIMIT 10")

print(f"получено строк:    {len(rows)}")
print(f"значений в строке: {len(rows[0]) if rows else 0}")

# Сырая строка из fetchall() — так видно, что вернула база, до сборки DataFrame
if rows:
    print(f"первая строка:     {rows[0][:6]}")

запрос не выполнен: UndefinedTable
сообщение сервера: relation "rnc_database" does not exist
получено строк:    0
значений в строке: 0


## 6. Сохранение полученных данных в DataFrame (1 балл)

In [7]:
# cursor.description хранит метаданные последнего запроса: первый элемент каждой
# записи — имя столбца. Без него DataFrame получил бы безымянные колонки 0, 1, 2...
databases = pd.DataFrame(rows, columns=columns)

print(f"строк: {databases.shape[0]}, столбцов: {databases.shape[1]}")
print()
if databases.empty:
    print("данные не получены — см. сообщение сервера в предыдущей ячейке")
else:
    databases[["id", "display_name", "num_sequences", "num_organisms", "alive"]]

строк: 0, столбцов: 0

данные не получены — см. сообщение сервера в предыдущей ячейке


## 7. Столбцы `display_name`, `num_sequences`, `num_organisms`, `url` (2 балла)

In [8]:
rows_selected, columns_selected, db_error = run_query("""
    SELECT display_name, num_sequences, num_organisms, url
    FROM rnc_database
    LIMIT 10
""")

selected = pd.DataFrame(rows_selected, columns=columns_selected)

print(f"строк: {selected.shape[0]}, столбцов: {selected.shape[1]}")
print()
if selected.empty:
    print("данные не получены — сообщение сервера смотрите в пункте 5")
else:
    selected

запрос не выполнен: UndefinedTable
сообщение сервера: relation "rnc_database" does not exist
строк: 0, столбцов: 0

данные не получены — сообщение сервера смотрите в пункте 5


## 8. Закрытие соединения

In [9]:
cursor.close()
connection.close()

print("курсор закрыт:", cursor.closed)
print("соединение закрыто:", connection.closed == 1)

курсор закрыт: True
соединение закрыто: True


### Итог

**API.** Получены 12 книг, 444 дома Вестероса и 68 домов с девизом (`hasWords=true`).
Фильтр `hasTitles=true`, который предложен в подсказке к заданию, отбирает 165 домов — и у 124
из них поле `words` пустое, то есть девиза нет: этот параметр фильтрует титулы, а не девиз.

**База данных.** Подключение к публичной базе RNAcentral через `psycopg2` работает: сервер
PostgreSQL 16, база `pfmegrnargs`, роль `reader`. Запросы к таблице `rnc_database` составлены
(полная выборка из 10 строк и выборка столбцов `display_name, num_sequences, num_organisms, url`),
но на момент запуска 09.10.2026 сервер EBI отказывает в чтении: `permission denied for schema
rnacen`. Ноутбук показывает ответ сервера в ячейках 5–7, поэтому запуск проходит без ошибок,
а данные появятся в тех же таблицах, как только доступ вернут.